# Lab 10 — Qiskit Primitives V2: StatevectorSampler y StatevectorEstimator

Qiskit 2.x unifica el acceso al hardware y los simuladores mediante dos primitivas:
- **StatevectorSampler**: mide distribuciones de probabilidad (cuasi-distribuciones)
- **StatevectorEstimator**: calcula valores esperados ⟨ψ|O|ψ⟩

Ambas aceptan circuitos parametrizados y permiten **batch evaluation** eficiente.

## 1. StatevectorSampler: distribuciones de medida

El Sampler ejecuta un circuito y retorna la distribución de las medidas.
Con `shots=None` calcula la distribución exacta; con shots finitos simula ruido estadístico.

In [ ]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives import StatevectorSampler, StatevectorEstimator
import numpy as np
import matplotlib.pyplot as plt

sampler = StatevectorSampler()

# Estado GHZ de 3 qubits
qc_ghz = QuantumCircuit(3)
qc_ghz.h(0)
qc_ghz.cx(0, 1)
qc_ghz.cx(1, 2)
qc_ghz.measure_all()

# Distribución exacta (modo statevector)
job_exact = sampler.run([qc_ghz], shots=10000)
counts = job_exact.result()[0].data.meas.get_counts()

print('Distribución GHZ |000⟩ + |111⟩ (10000 shots):')
for bitstr, cnt in sorted(counts.items(), key=lambda x: -x[1]):
    bar = '█' * int(cnt / 200)
    print(f'  |{bitstr}⟩: {cnt:5d}  {bar}')

## 2. Circuitos parametrizados con el Sampler

El Sampler puede evaluar circuitos con parámetros en batch.
Esto es esencial para VQE/QAOA donde se barren muchos puntos del espacio de parámetros.

In [ ]:
from qiskit.circuit import ParameterVector

# Circuito parametrizado: estado de Bell generalizado
# |ψ(θ)⟩ = cos(θ/2)|00⟩ + sin(θ/2)|11⟩
theta = ParameterVector('θ', 1)
qc_param = QuantumCircuit(2)
qc_param.ry(theta[0], 0)
qc_param.cx(0, 1)
qc_param.measure_all()

# Batch: evaluar para 20 valores de theta
theta_values = np.linspace(0, np.pi, 20)
batch = [(qc_param, [t]) for t in theta_values]
job_batch = sampler.run(batch, shots=2000)

p11 = []
for i, result in enumerate(job_batch.result()):
    counts = result.data.meas.get_counts()
    total = sum(counts.values())
    p11.append(counts.get('11', 0) / total)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(theta_values / np.pi, p11, 'bo-', markersize=5, label='P(|11⟩) simulado')
ax.plot(theta_values / np.pi, np.sin(theta_values/2)**2, 'r--', label='sin²(θ/2) teórico')
ax.set_xlabel('θ / π', fontsize=12)
ax.set_ylabel('P(|11⟩)', fontsize=12)
ax.set_title('Sampler batch: estado de Bell generalizado', fontsize=13)
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 3. StatevectorEstimator: valores esperados

El Estimator calcula ⟨ψ(θ)|O|ψ(θ)⟩ directamente, sin necesidad de medir en bases rotadas.
Es el motor de los algoritmos variacionales: VQE, QAOA, QNN.

In [ ]:
estimator = StatevectorEstimator()

# Circuito parametrizado sin medición
qc_est = QuantumCircuit(2)
qc_est.ry(theta[0], 0)
qc_est.cx(0, 1)

# Observables a medir
observables = {
    'ZZ': SparsePauliOp('ZZ'),
    'XX': SparsePauliOp('XX'),
    'ZI': SparsePauliOp('ZI'),
    'IZ': SparsePauliOp('IZ'),
}

# Evaluar todos los observables en θ = π/2 (estado de Bell aproximado)
t_pi2 = np.pi / 2
print(f'Valores esperados para θ = π/2 (estado ~Bell):')
for name, obs in observables.items():
    job = estimator.run([(qc_est, obs, [t_pi2])])
    val = float(job.result()[0].data.evs)
    print(f'  ⟨{name}⟩ = {val:+.4f}')

## 4. Comparación: Sampler vs Estimator para medir ⟨Z⟩

Para medir ⟨Z⟩ con el Sampler se necesita construir el estimador manualmente a partir
de las cuentas de |0⟩ y |1⟩. El Estimator lo hace directamente y es más preciso.

In [ ]:
# Medir ⟨Z⟩ de Ry(θ)|0⟩ con ambas primitivas
thetas = np.linspace(0, 2*np.pi, 30)
z_sampler = []
z_estimator = []

qc_z = QuantumCircuit(1)
qc_z.ry(theta[0], 0)
qc_z_meas = qc_z.copy()
qc_z_meas.measure_all()

H_z = SparsePauliOp('Z')

# Batch Sampler
batch_s = [(qc_z_meas, [t]) for t in thetas]
results_s = sampler.run(batch_s, shots=4000).result()
for r in results_s:
    c = r.data.meas.get_counts()
    total = sum(c.values())
    z_sampler.append((c.get('0', 0) - c.get('1', 0)) / total)

# Batch Estimator
batch_e = [(qc_z, H_z, [t]) for t in thetas]
results_e = estimator.run(batch_e).result()
z_estimator = [float(r.data.evs) for r in results_e]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(thetas/np.pi, z_sampler, 'b.', markersize=6, label='Sampler (4000 shots)')
ax.plot(thetas/np.pi, z_estimator, 'r-', linewidth=2, label='Estimator (exacto)')
ax.plot(thetas/np.pi, np.cos(thetas), 'k--', alpha=0.4, label='cos(θ) teórico')
ax.set_xlabel('θ / π', fontsize=12)
ax.set_ylabel('⟨Z⟩', fontsize=12)
ax.set_title('⟨Z⟩ de Ry(θ)|0⟩: Sampler vs Estimator', fontsize=13)
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Cuándo usar cada primitiva

| Primitiva | Cuándo usarla | Ventaja |
|-----------|--------------|--------|
| **Sampler** | Algoritmos de muestreo, resultados discretos, QAOA final | Distribución completa |
| **Estimator** | VQE, gradientes, valores esperados de H | Exacto, sin ruido estadístico |

Para hardware real, ambas primitivas abstraen la mitigación de errores (M3, ZNE, PEC)
a través del parámetro `resilience_level` de `EstimatorV2`/`SamplerV2` en `qiskit_ibm_runtime`.

In [ ]:
# Ejemplo: uso típico en VQE con Estimator
from scipy.optimize import minimize

H_heisenberg = SparsePauliOp.from_list([('XX', 1.0), ('YY', 1.0), ('ZZ', 1.0)])
E0_exact = np.linalg.eigvalsh(H_heisenberg.to_matrix())[0]

theta2 = ParameterVector('θ', 4)
ansatz = QuantumCircuit(2)
ansatz.ry(theta2[0], 0); ansatz.ry(theta2[1], 1)
ansatz.cx(0, 1)
ansatz.ry(theta2[2], 0); ansatz.ry(theta2[3], 1)

n_evals = [0]
def cost(params):
    n_evals[0] += 1
    return float(estimator.run([(ansatz, H_heisenberg, params)]).result()[0].data.evs)

np.random.seed(1)
res = minimize(cost, np.random.uniform(-np.pi, np.pi, 4), method='COBYLA',
               options={'maxiter': 400})
print(f'VQE con Estimator:')
print(f'  E encontrada:  {res.fun:.6f}')
print(f'  E₀ exacta:     {E0_exact:.6f}')
print(f'  Error:         {abs(res.fun - E0_exact):.2e}')
print(f'  Evaluaciones:  {n_evals[0]}')